### Section 1: Import Required Libraries and Set Up


In [2]:
# Import necessary libraries
import pandas as pd
import os
from pathlib import Path

# Set the directory path
directory_path = r'V:\Recherchen\LBDB_Bilderlisten\ProcessingForMaßstabGeneration\xlsxCopy'

print("🔧 Libraries imported and path set")

🔧 Libraries imported and path set


### Section 2: Access All XLSX Files and Count Them


In [3]:
# Get ALL Excel files from the directory (including different formats)
excel_extensions = ['.xlsx', '.xls', '.xlsm', '.xlsb']  # All common Excel formats
excel_files = []

print("🔍 Scanning directory for Excel files...")

# Walk through all files in the directory and subdirectories
for root, dirs, files in os.walk(directory_path):
    for file in files:
        file_lower = file.lower()
        if any(file_lower.endswith(ext) for ext in excel_extensions):
            full_path = os.path.join(root, file)
            excel_files.append(full_path)

# Alternative: If you only want files in the main directory (not subfolders)
# excel_files = [os.path.join(directory_path, f) for f in os.listdir(directory_path) 
#               if any(f.lower().endswith(ext) for ext in excel_extensions)]

# Print status and count
print(f"📁 Total Excel files found: {len(excel_files)}")
print("Files found:")
for i, file in enumerate(excel_files, 1):
    print(f"{i}. {os.path.basename(file)} (Path: {file})")

# Store the count for later reference
total_excel_files = len(excel_files)

🔍 Scanning directory for Excel files...
📁 Total Excel files found: 76
Files found:
1. 2015-12-01 Hamlar - Rain Bilderliste.xlsx (Path: V:\Recherchen\LBDB_Bilderlisten\ProcessingForMaßstabGeneration\xlsxCopy\2015-12-01 Hamlar - Rain Bilderliste.xlsx)
2. Alzey 16041841_HINKELBEIN.xlsx (Path: V:\Recherchen\LBDB_Bilderlisten\ProcessingForMaßstabGeneration\xlsxCopy\Alzey 16041841_HINKELBEIN.xlsx)
3. Annweiler am Trifels 16042143_HINKELBEIN.xlsx (Path: V:\Recherchen\LBDB_Bilderlisten\ProcessingForMaßstabGeneration\xlsxCopy\Annweiler am Trifels 16042143_HINKELBEIN.xlsx)
4. B3 Malsch bis Ubstadt-Weiher 180208702_HINKELBEIN.XLSX (Path: V:\Recherchen\LBDB_Bilderlisten\ProcessingForMaßstabGeneration\xlsxCopy\B3 Malsch bis Ubstadt-Weiher 180208702_HINKELBEIN.XLSX)
5. Bad Bergzabern 16030193_HINKELBEIN.xlsx (Path: V:\Recherchen\LBDB_Bilderlisten\ProcessingForMaßstabGeneration\xlsxCopy\Bad Bergzabern 16030193_HINKELBEIN.xlsx)
6. Baustert Trasse westlich Bitburg 180903550_HINKELBEIN.xlsx (Path: V:\Re

### Section 3: Function to Process Individual Files


In [4]:
def process_excel_file(file_path, expected_headers=['Sortie', 'BildNr', 'Datum', 'Masstab']):
    """
    Process a single Excel file: skip first 2 rows, check for required headers,
    and extract specified columns
    """
    try:
        # Read the file, skipping first 2 rows (so row 3 becomes header)
        df = pd.read_excel(file_path, skiprows=2)
        
        # Check if required headers exist
        missing_headers = [header for header in expected_headers if header not in df.columns]
        
        if missing_headers:
            print(f"❌ Missing headers in {os.path.basename(file_path)}: {missing_headers}")
            return None
        else:
            print(f"✅ All required headers found in {os.path.basename(file_path)}")
            
            # Select only the required columns
            selected_columns = df[expected_headers].copy()
            
            # Add source file name for reference
            selected_columns['Source_File'] = os.path.basename(file_path)
            
            return selected_columns
            
    except Exception as e:
        print(f"❌ Error processing {os.path.basename(file_path)}: {str(e)}")
        return None

### Modified Section 4: Process All Files with Full Paths


In [5]:
# Enhanced version with more detailed statistics
print("\n🔄 Processing files with detailed statistics...")
print("-" * 70)

all_dataframes = []
processing_stats = {
    'total_files': len(excel_files),
    'successful': 0,
    'failed': 0,
    'total_rows': 0,
    'files_with_missing_headers': 0,
    'files_empty': 0,
    'files_with_errors': 0,
    'rows_per_file': []
}

for file_path in excel_files:
    file_name = os.path.basename(file_path)
    print(f"\n📄 Processing: {file_name}")
    
    try:
        # Read the file
        df = pd.read_excel(file_path, skiprows=2)
        
        if df.empty:
            print(f"   ⚠️  File is empty")
            processing_stats['files_empty'] += 1
            processing_stats['failed'] += 1
            continue
        
        # Case-insensitive header matching
        df_columns_lower = [str(col).lower().strip() for col in df.columns]
        header_mapping = {}
        missing_headers = []
        
        for expected_header in ['Sortie', 'BildNr', 'Datum', 'Masstab']:
            expected_lower = expected_header.lower()
            if expected_lower in df_columns_lower:
                actual_index = df_columns_lower.index(expected_lower)
                header_mapping[expected_header] = df.columns[actual_index]
            else:
                missing_headers.append(expected_header)
        
        if missing_headers:
            print(f"   ❌ Missing headers: {missing_headers}")
            processing_stats['files_with_missing_headers'] += 1
            processing_stats['failed'] += 1
            continue
        
        # Extract data
        actual_cols = [header_mapping[h] for h in ['Sortie', 'BildNr', 'Datum', 'Masstab']]
        selected_data = df[actual_cols].copy()
        selected_data.columns = ['Sortie', 'BildNr', 'Datum', 'Masstab']
        
        # Add source info
        selected_data['Source_File'] = file_name
        
        # Update statistics
        rows_in_file = len(selected_data)
        processing_stats['total_rows'] += rows_in_file
        processing_stats['rows_per_file'].append(rows_in_file)
        processing_stats['successful'] += 1
        
        all_dataframes.append(selected_data)
        
        print(f"   ✅ Extracted {rows_in_file} rows")
        
    except Exception as e:
        print(f"   ❌ Error: {str(e)}")
        processing_stats['files_with_errors'] += 1
        processing_stats['failed'] += 1

# Print comprehensive summary
print("\n" + "="*80)
print("📈 DETAILED PROCESSING SUMMARY")
print("="*80)
print(f"Total Excel files scanned: {processing_stats['total_files']:,}")
print(f"Successfully processed: {processing_stats['successful']:,} files")
print(f"Failed to process: {processing_stats['failed']:,} files")
print(f"  - Due to missing headers: {processing_stats['files_with_missing_headers']:,}")
print(f"  - Empty files: {processing_stats['files_empty']:,}")
print(f"  - Other errors: {processing_stats['files_with_errors']:,}")
print(f"Total rows extracted: {processing_stats['total_rows']:,}")
print(f"Success rate: {(processing_stats['successful']/processing_stats['total_files']*100):.1f}%")

if processing_stats['successful'] > 0:
    avg_rows = processing_stats['total_rows'] / processing_stats['successful']
    min_rows = min(processing_stats['rows_per_file'])
    max_rows = max(processing_stats['rows_per_file'])
    
    print(f"Average rows per file: {avg_rows:.1f}")
    print(f"Minimum rows in a file: {min_rows}")
    print(f"Maximum rows in a file: {max_rows}")
    print(f"Total data points: {processing_stats['total_rows'] * 4:,} (4 columns × {processing_stats['total_rows']:,} rows)")

print("="*80)


🔄 Processing files with detailed statistics...
----------------------------------------------------------------------

📄 Processing: 2015-12-01 Hamlar - Rain Bilderliste.xlsx
   ✅ Extracted 47 rows

📄 Processing: Alzey 16041841_HINKELBEIN.xlsx
   ✅ Extracted 102 rows

📄 Processing: Annweiler am Trifels 16042143_HINKELBEIN.xlsx
   ✅ Extracted 164 rows

📄 Processing: B3 Malsch bis Ubstadt-Weiher 180208702_HINKELBEIN.XLSX
   ✅ Extracted 441 rows

📄 Processing: Bad Bergzabern 16030193_HINKELBEIN.xlsx
   ✅ Extracted 120 rows

📄 Processing: Baustert Trasse westlich Bitburg 180903550_HINKELBEIN.xlsx
   ✅ Extracted 88 rows

📄 Processing: Bernkastel-Kues 16020342_HINKELBEIN.xlsx
   ✅ Extracted 100 rows

📄 Processing: Bescheid 16091942_HINKELBEIN.XLSX
   ✅ Extracted 38 rows

📄 Processing: Bettingen 170116505_HINKELBEIN.XLSX
   ✅ Extracted 31 rows

📄 Processing: Biberach an der Riß 16040590_HINKELBEIN.xlsx
   ✅ Extracted 197 rows

📄 Processing: Bornheim 15120452_HINKELBEIN.xlsx
   ✅ Extracted 67

### Section 5: Merge All Data and Create Final Excel File


In [ ]:
if all_dataframes:
    # Merge all dataframes
    merged_df = pd.concat(all_dataframes, ignore_index=True)
    
    # Print summary of merged data
    print(f"\n📈 Merged data shape: {merged_df.shape}")
    print(f"📊 Total rows in merged data: {len(merged_df)}")
    
    # Display first few rows
    print("\n🔍 First 5 rows of merged data:")
    display(merged_df.head())
    
    """ # Save to Excel file
    output_path = os.path.join(directory_path, 'Merged.xlsx')
    merged_df.to_excel(output_path, index=False)
    
    print(f"\n💾 Successfully saved merged file to: {output_path}")
    print("✅ Process completed successfully!")  """
    
else:
    print("❌ No files were successfully processed. Cannot create merged file.")


📈 Merged data shape: (16991, 5)
📊 Total rows in merged data: 16991

🔍 First 5 rows of merged data:


,Sortie,BildNr,Datum,Masstab,Source_File
0,7-4013,4097.0,1945-02-25,12500.0,2015-12-01 Hamlar - Rain Bilderliste.xlsx
1,7-4013,7044.0,1945-02-25,50000.0,2015-12-01 Hamlar - Rain Bilderliste.xlsx
2,7-4014,7015.0,1945-02-25,50000.0,2015-12-01 Hamlar - Rain Bilderliste.xlsx
3,7-4014,7016.0,1945-02-25,50000.0,2015-12-01 Hamlar - Rain Bilderliste.xlsx
4,104W-042C,4131.0,1945-04-08,8300.0,2015-12-01 Hamlar - Rain Bilderliste.xlsx



💾 Successfully saved merged file to: V:\Recherchen\LBDB_Bilderlisten\ProcessingForMaßstabGeneration\xlsxCopy\Merged.xlsx
✅ Process completed successfully!


### Section 6: Additional Verification and Summary (Not neccesary if the Merged excel not generated)


In [8]:
""" # Optional: Verify the merged file was created correctly
if all_dataframes:
    # Read the merged file to verify
    verify_df = pd.read_excel(output_path)
    
    print("\n🔍 Verification of merged file:")
    print(f"File exists: {os.path.exists(output_path)}")
    print(f"Rows in merged file: {len(verify_df)}")
    print(f"Columns in merged file: {list(verify_df.columns)}")
    
    # Count rows per source file
    print("\n📊 Rows per source file:")
    file_counts = verify_df['Source_File'].value_counts()
    print(file_counts)
    
    # Check for any missing values in critical columns
    print("\n🔍 Missing values in critical columns:")
    for col in ['Sortie', 'BildNr', 'Datum', 'Masstab']:
        missing_count = verify_df[col].isnull().sum()
        print(f"{col}: {missing_count} missing values") """

' # Optional: Verify the merged file was created correctly\nif all_dataframes:\n    # Read the merged file to verify\n    verify_df = pd.read_excel(output_path)\n\n    print("\n🔍 Verification of merged file:")\n    print(f"File exists: {os.path.exists(output_path)}")\n    print(f"Rows in merged file: {len(verify_df)}")\n    print(f"Columns in merged file: {list(verify_df.columns)}")\n\n    # Count rows per source file\n    print("\n📊 Rows per source file:")\n    file_counts = verify_df[\'Source_File\'].value_counts()\n    print(file_counts)\n\n    # Check for any missing values in critical columns\n    print("\n🔍 Missing values in critical columns:")\n    for col in [\'Sortie\', \'BildNr\', \'Datum\', \'Masstab\']:\n        missing_count = verify_df[col].isnull().sum()\n        print(f"{col}: {missing_count} missing values") '

### Section 7: Read Excel File 'Bilderbestellungen_Fluginfos' and Extract Sortie Values


In [14]:
# Enhanced version with more detailed analysis
print("🔍 Section 7: Detailed Excel File Analysis")
print("="*80)

file_path = r'L:\_Luftbilddatenbank\ExcelHerkunft\Bilderbestellungen_Fluginfos.xlsm'
sheet_name = 'DB_Befliegungen'

try:
    # Read the file with more options
    df = pd.read_excel(file_path, 
                      sheet_name=sheet_name,
                      engine='openpyxl')  # Specify engine for .xlsm files
    
    print(f"✅ File loaded successfully")
    print(f"📊 DataFrame shape: {df.shape}")
    print(f"📝 Columns: {list(df.columns)}")
    
    # Case-insensitive column search
    target_col = None
    for col in df.columns:
        if str(col).lower() == 'sortie':
            target_col = col
            break
    
    if target_col is None:
        print("❌ 'Sortie' column not found!")
        print("Available columns:")
        for col in df.columns:
            print(f"  - {col} (type: {type(col)})")
        raise ValueError("Sortie column not found")
    
    print(f"✅ Found Sortie column: '{target_col}'")
    
    # Extract and analyze Sortie values
    sortie_data = df[target_col].copy()
    
    print(f"\n📈 Sortie Column Analysis:")
    print(f"Total rows: {len(sortie_data)}")
    print(f"Non-null values: {sortie_data.notna().sum()}")
    print(f"Null values: {sortie_data.isna().sum()}")
    print(f"Unique values: {sortie_data.nunique()}")
    
    # Get unique values (excluding NaN)
    unique_sorties = sortie_data.dropna().unique()
    print(f"\n🎯 Unique Sortie values: {len(unique_sorties)}")
    
    # Display values in a nice format
    print(f"\n📋 Sortie Values Sample:")
    if len(unique_sorties) > 0:
        # Convert to sorted list for better display
        try:
            sorted_sorties = sorted(unique_sorties, key=lambda x: str(x))
        except:
            sorted_sorties = unique_sorties
        
        for i, value in enumerate(sorted_sorties[:20], 1):
            print(f"  {i:2d}. {value}")
        
        if len(unique_sorties) > 20:
            print(f"  ... and {len(unique_sorties) - 20} more values")
    else:
        print("  No Sortie values found")
    
    # Data type analysis
    print(f"\n🔍 Data Type Analysis:")
    print(f"Column dtype: {sortie_data.dtype}")
    
    # Value distribution
    if len(unique_sorties) > 0:
        value_counts = sortie_data.value_counts().head(10)
        print(f"\n📊 Top 10 most frequent Sortie values:")
        for value, count in value_counts.items():
            print(f"  {value}: {count} occurrences")
    
    # Store for later use
    global extracted_sortie_values
    extracted_sortie_values = unique_sorties
    print(f"\n💾 Values stored in variable: extracted_sortie_values")

except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    print(f"Detailed error: {traceback.format_exc()}")

print("="*80)

🔍 Section 7: Detailed Excel File Analysis
✅ File loaded successfully
📊 DataFrame shape: (21718, 27)
📝 Columns: ['Geprüft (Ja / Nein)', 'Eingetragen in GIS/Bearbeiter', 'Sortie', 'FlugnummerLGL', 'Datum', 'NCAP Finding Aid/NARA Spot Nr.', 'Zusätzliche NCAP Finding Aid/NARA Spot Nr.', 'Herkunft', 'Quelle Sortie, Anmerkungen', 'Maßstab', 'Maßstab B.-Nr. 0000', 'Maßstab B.-Nr. 1000', 'Maßstab B.-Nr. 2000', 'Maßstab B.-Nr. 3000', 'Maßstab B.-Nr. 4000', 'Maßstab B.-Nr. 5000', 'Maßstab B.-Nr. 6000', 'Maßstab B.-Nr. 7000', 'Maßstab B.-Nr. 8000', 'Maßstab B.-Nr. >9000', 'Flugnummer Bayern', 'Flugnummer RLP', 'Qualität', 'Ohne Bildmitte oder Eintrag in der DB', 'Flughöhe (in feet)', 'Brennweite (Focal Length, in inch)', 'Time']
✅ Found Sortie column: 'Sortie'

📈 Sortie Column Analysis:
Total rows: 21718
Non-null values: 21667
Null values: 51
Unique values: 21575

🎯 Unique Sortie values: 21575

📋 Sortie Values Sample:
   1. 0001_AMS
   2. 0065_4X_31
   3. 0415_243
   4. 1 RIAF 574
   5. 1 RIAF 57

### Section 8: Cross-Check and Match Sortie Values


In [15]:
# Section 8: Cross-Check and Match Sortie Values
print("🔍 Section 8: Cross-checking Sortie values between datasets")
print("="*80)

# First, let's ensure we have both datasets
print("📊 Dataset Overview:")
print(f"Merged DataFrame shape: {merged_df.shape}")
print(f"Extracted Sortie values count: {len(extracted_sortie_values)}")

# Display sample values from both sources
print("\n📋 Sample values from Merged DataFrame:")
print(merged_df['Sortie'].head(10).tolist())

print("\n📋 Sample values from BilderbestellungenFluginfos:")
print(extracted_sortie_values[:10].tolist())

🔍 Section 8: Cross-checking Sortie values between datasets
📊 Dataset Overview:
Merged DataFrame shape: (16991, 5)
Extracted Sortie values count: 21575

📋 Sample values from Merged DataFrame:
['7-4013', '7-4013', '7-4014', '7-4014', '104W-042C', '106G-1657', '106G-1657', '106G-1657', '106G-1657', '32-0439']

📋 Sample values from BilderbestellungenFluginfos:
['TK25_6616', 'TU_GX_01844', 'TK25_6118', 'TK25_6512', 'TK_2609', 'TK25_6719', 'FRENCH MISSION 388', 'XA_0001', 'TK_2510', 'TU_GX_02223']


#### Revised Section 8.1: Simplified Matching Approach



In [16]:
# Section 8.1: Simplified Matching Approach
print("\n🔄 Simplified Sortie matching based on alphanumeric content...")

import re
from difflib import SequenceMatcher

def extract_alphanumeric_content(sortie_value):
    """
    Extract only the alphanumeric characters (ignore underscores, dashes, etc.)
    """
    if pd.isna(sortie_value) or sortie_value == "":
        return ""
    
    # Convert to uppercase and remove all non-alphanumeric characters
    s = str(sortie_value).upper()
    alphanumeric_only = re.sub(r'[^A-Z0-9]', '', s)
    
    return alphanumeric_only

def calculate_alphanumeric_similarity(str1, str2):
    """
    Calculate similarity based on alphanumeric content only
    """
    if pd.isna(str1) or pd.isna(str2) or str1 == "" or str2 == "":
        return 0.0
    
    # Extract alphanumeric content
    alpha1 = extract_alphanumeric_content(str1)
    alpha2 = extract_alphanumeric_content(str2)
    
    if alpha1 == "" or alpha2 == "":
        return 0.0
    
    # Calculate similarity
    similarity = SequenceMatcher(None, alpha1, alpha2).ratio() * 100
    
    return similarity

def find_simple_match(source_value, target_values, threshold=70.0):
    """
    Simple matching based on alphanumeric content similarity
    """
    if pd.isna(source_value) or source_value == "":
        return None, 0.0
    
    source_alpha = extract_alphanumeric_content(source_value)
    
    best_match = None
    best_similarity = 0.0
    
    for target_value in target_values:
        if pd.isna(target_value) or target_value == "":
            continue
            
        target_alpha = extract_alphanumeric_content(target_value)
        
        # Skip if either string has no alphanumeric content
        if source_alpha == "" or target_alpha == "":
            continue
        
        # Calculate similarity
        similarity = calculate_alphanumeric_similarity(source_value, target_value)
        
        if similarity > best_similarity and similarity >= threshold:
            best_similarity = similarity
            best_match = target_value
    
    return best_match, best_similarity

# Test the simplified functions with your examples
test_cases = [
    ("104W_035C", "104W_C_0035"),  # Should match: 104W035C vs 104WC0035
    ("31_4788", "US31_4788"),      # Should match: 314788 vs US314788  
    ("7_156A", "US7_0156_A"),      # Should match: 7156A vs US70156A
    ("NO_MATCH", "NON_EXISTENT")   # Should not match
]

print("\n🧪 Testing simplified matching function:")
for source, target in test_cases:
    source_alpha = extract_alphanumeric_content(source)
    target_alpha = extract_alphanumeric_content(target)
    
    match, similarity = find_simple_match(source, [target], threshold=50.0)
    
    print(f"  {source} -> {target}")
    print(f"    Alphanumeric: '{source_alpha}' vs '{target_alpha}'")
    print(f"    Similarity: {similarity:.1f}%")
    print(f"    Match: {'✅' if match is not None else '❌'}")
    print()


🔄 Simplified Sortie matching based on alphanumeric content...

🧪 Testing simplified matching function:
  104W_035C -> 104W_C_0035
    Alphanumeric: '104W035C' vs '104WC0035'
    Similarity: 82.4%
    Match: ✅

  31_4788 -> US31_4788
    Alphanumeric: '314788' vs 'US314788'
    Similarity: 85.7%
    Match: ✅

  7_156A -> US7_0156_A
    Alphanumeric: '7156A' vs 'US70156A'
    Similarity: 76.9%
    Match: ✅

  NO_MATCH -> NON_EXISTENT
    Alphanumeric: 'NOMATCH' vs 'NONEXISTENT'
    Similarity: 0.0%
    Match: ❌



#### Revised Section 8.2: Perform Cross-Checking with Simple Matching on All data

In [17]:
# Section 8.2: Perform cross-checking with simple matching and print matches
print("\n🔄 Cross-checking all Sortie values with simple matching...")

# Convert extracted values to list for matching
target_sorties = [x for x in extracted_sortie_values if pd.notna(x) and x != ""]

print(f"Target Sorties available for matching: {len(target_sorties)}")

# Create lists to store results
matched_values = []
similarity_scores = []

total_rows = len(merged_df)
match_count = 0
first_50_matches = []  # Store first 50 matches for printing

print(f"Processing {total_rows} rows...")

for idx, row in merged_df.iterrows():
    source_sortie = row['Sortie']
    
    # Find best match with lower threshold to catch more matches
    best_match, similarity = find_simple_match(source_sortie, target_sorties, threshold=50.0)
    
    matched_values.append(best_match)
    similarity_scores.append(similarity)
    
    if best_match is not None:
        match_count += 1
        # Store first 50 matches
        if match_count <= 50:
            first_50_matches.append({
                'source': source_sortie,
                'target': best_match,
                'similarity': similarity,
                'source_alpha': extract_alphanumeric_content(source_sortie),
                'target_alpha': extract_alphanumeric_content(best_match)
            })
    
    # Progress update every 100 rows
    if (idx + 1) % 100 == 0:
        print(f"  Processed {idx + 1}/{total_rows} rows, "
              f"matches found: {match_count} ({match_count/(idx+1)*100:.1f}%)")
        
        # Print first 50 matches after processing first 100 rows
        if idx + 1 == 100 and first_50_matches:
            print(f"\n📋 First {len(first_50_matches)} matched Sorties:")
            print("-" * 80)
            for i, match in enumerate(first_50_matches, 1):
                print(f"{i:2d}. {match['source']} -> {match['target']}")
                print(f"    Similarity: {match['similarity']:.1f}%")
                print(f"    Alphanumeric: '{match['source_alpha']}' vs '{match['target_alpha']}'")
                print()
    
    # Final update
    if (idx + 1) == total_rows:
        print(f"  Processed {idx + 1}/{total_rows} rows, "
              f"matches found: {match_count} ({match_count/(idx+1)*100:.1f}%)")

print(f"\n✅ Cross-checking completed!")

# Print all first 50 matches if we didn't print them already
if first_50_matches and total_rows < 100:
    print(f"\n📋 First {len(first_50_matches)} matched Sorties:")
    print("-" * 80)
    for i, match in enumerate(first_50_matches, 1):
        print(f"{i:2d}. {match['source']} -> {match['target']}")
        print(f"    Similarity: {match['similarity']:.1f}%")
        print(f"    Alphanumeric: '{match['source_alpha']}' vs '{match['target_alpha']}'")
        print()


🔄 Cross-checking all Sortie values with simple matching...
Target Sorties available for matching: 21575
Processing 16991 rows...
  Processed 100/16991 rows, matches found: 100 (100.0%)

📋 First 50 matched Sorties:
--------------------------------------------------------------------------------
 1. 7-4013 -> US7_4013
    Similarity: 83.3%
    Alphanumeric: '74013' vs 'US74013'

 2. 7-4013 -> US7_4013
    Similarity: 83.3%
    Alphanumeric: '74013' vs 'US74013'

 3. 7-4014 -> US7_4014
    Similarity: 83.3%
    Alphanumeric: '74014' vs 'US74014'

 4. 7-4014 -> US7_4014
    Similarity: 83.3%
    Alphanumeric: '74014' vs 'US74014'

 5. 104W-042C -> 104W_C_0042
    Similarity: 82.4%
    Alphanumeric: '104W042C' vs '104WC0042'

 6. 106G-1657 -> 106G_1657
    Similarity: 100.0%
    Alphanumeric: '106G1657' vs '106G1657'

 7. 106G-1657 -> 106G_1657
    Similarity: 100.0%
    Alphanumeric: '106G1657' vs '106G1657'

 8. 106G-1657 -> 106G_1657
    Similarity: 100.0%
    Alphanumeric: '106G1657' v

#### Revised Section 8.3: Add Results to DataFrame


In [18]:
# Section 8.3: Add results to the merged DataFrame
print("\n📊 Adding results to merged DataFrame...")

# Add the new columns
merged_df['Sortie_BilderbestellungenFluginfos'] = matched_values
merged_df['Similarity_Percentage'] = similarity_scores

# Create confidence categories based on similarity
def get_confidence(similarity):
    if pd.isna(similarity):
        return 'No Match'
    elif similarity >= 90:
        return 'Very High'
    elif similarity >= 80:
        return 'High'
    elif similarity >= 70:
        return 'Medium'
    elif similarity >= 60:
        return 'Low'
    elif similarity >= 50:
        return 'Very Low'
    else:
        return 'No Match'

merged_df['Match_Confidence'] = [get_confidence(sim) for sim in similarity_scores]

# Add alphanumeric versions for reference
merged_df['Source_Alphanumeric'] = [extract_alphanumeric_content(x) for x in merged_df['Sortie']]
merged_df['Target_Alphanumeric'] = [extract_alphanumeric_content(x) if pd.notna(x) else "" for x in merged_df['Sortie_BilderbestellungenFluginfos']]

# Calculate match status
merged_df['Match_Found'] = merged_df['Sortie_BilderbestellungenFluginfos'].notna()

print("✅ Columns added to DataFrame")
print(f"DataFrame shape: {merged_df.shape}")
print(f"New columns: {list(merged_df.columns[-6:])}")  # Show the last 6 added columns

# Quick preview of the results
print(f"\n🔍 Preview of matched data:")
preview_df = merged_df[['Sortie', 'Sortie_BilderbestellungenFluginfos', 'Similarity_Percentage', 'Match_Confidence']].head(10)
display(preview_df)


📊 Adding results to merged DataFrame...
✅ Columns added to DataFrame
DataFrame shape: (16991, 11)
New columns: ['Sortie_BilderbestellungenFluginfos', 'Similarity_Percentage', 'Match_Confidence', 'Source_Alphanumeric', 'Target_Alphanumeric', 'Match_Found']

🔍 Preview of matched data:


,Sortie,Sortie_BilderbestellungenFluginfos,Similarity_Percentage,Match_Confidence
0,7-4013,US7_4013,83.333333,High
1,7-4013,US7_4013,83.333333,High
2,7-4014,US7_4014,83.333333,High
3,7-4014,US7_4014,83.333333,High
4,104W-042C,104W_C_0042,82.352941,High
5,106G-1657,106G_1657,100.000000,Very High
6,106G-1657,106G_1657,100.000000,Very High
7,106G-1657,106G_1657,100.000000,Very High
8,106G-1657,106G_1657,100.000000,Very High
9,32-0439,32S_0439,92.307692,Very High


#### Revised Section 8.4: Comprehensive Analysis and Results


In [19]:
# Section 8.4: Comprehensive Analysis and Results Summary
print("\n📈 COMPREHENSIVE MATCHING RESULTS ANALYSIS")
print("="*90)

total_rows = len(merged_df)
total_matches = merged_df['Match_Found'].sum()
no_matches = total_rows - total_matches

# Calculate matches by confidence level
confidence_stats = {}
for confidence_level in ['Very High', 'High', 'Medium', 'Low', 'Very Low', 'No Match']:
    count = len(merged_df[merged_df['Match_Confidence'] == confidence_level])
    confidence_stats[confidence_level] = count

# Calculate average similarity for matches
avg_similarity = merged_df.loc[merged_df['Match_Found'], 'Similarity_Percentage'].mean()

print(f"📊 OVERALL STATISTICS:")
print(f"Total rows processed: {total_rows:,}")
print(f"Rows with matches: {total_matches:,} ({total_matches/total_rows*100:.1f}%)")
print(f"Rows without matches: {no_matches:,} ({no_matches/total_rows*100:.1f}%)")
print(f"Average similarity for matches: {avg_similarity:.1f}%")

print(f"\n🎯 MATCH CONFIDENCE DISTRIBUTION:")
print("-" * 50)
for level in ['Very High', 'High', 'Medium', 'Low', 'Very Low']:
    count = confidence_stats[level]
    percentage = count / total_matches * 100 if total_matches > 0 else 0
    print(f"  {level:10}: {count:>5,} matches ({percentage:5.1f}% of matches)")

print(f"  {'No Match':10}: {no_matches:>5,} rows")

print(f"\n📈 SIMILARITY SCORE DISTRIBUTION:")
similarity_bins = pd.cut(merged_df['Similarity_Percentage'].fillna(0), 
                         bins=[0, 50, 60, 70, 80, 90, 100],
                         labels=['<50%', '50-60%', '60-70%', '70-80%', '80-90%', '90-100%'])
similarity_dist = similarity_bins.value_counts().sort_index()
for bin_label, count in similarity_dist.items():
    print(f"  {bin_label:8}: {count:>5,} rows")

# Show top matches by similarity
print(f"\n⭐ TOP 10 HIGHEST SIMILARITY MATCHES:")
top_matches = merged_df[merged_df['Match_Found']].nlargest(10, 'Similarity_Percentage')
for i, (idx, row) in enumerate(top_matches.iterrows(), 1):
    print(f"  {i:2d}. {row['Sortie']} -> {row['Sortie_BilderbestellungenFluginfos']}")
    print(f"      Similarity: {row['Similarity_Percentage']:.1f}%, Confidence: {row['Match_Confidence']}")
    print(f"      Alphanumeric: '{row['Source_Alphanumeric']}' vs '{row['Target_Alphanumeric']}'")

# Show some examples of different confidence levels
print(f"\n🔍 SAMPLE MATCHES BY CONFIDENCE LEVEL:")
confidence_samples = {}
for level in ['Very High', 'High', 'Medium', 'Low', 'Very Low']:
    level_matches = merged_df[merged_df['Match_Confidence'] == level]
    if len(level_matches) > 0:
        confidence_samples[level] = level_matches.head(2)

for level, sample_df in confidence_samples.items():
    print(f"\n{level} Confidence Examples:")
    for _, row in sample_df.iterrows():
        print(f"  {row['Sortie']} -> {row['Sortie_BilderbestellungenFluginfos']} ({row['Similarity_Percentage']:.1f}%)")

# Analyze non-matches
print(f"\n❌ ANALYSIS OF NON-MATCHES:")
non_match_samples = merged_df[~merged_df['Match_Found']].head(5)
print(f"Sample of {len(non_match_samples)} non-matching Sorties:")
for _, row in non_match_samples.iterrows():
    source_alpha = extract_alphanumeric_content(row['Sortie'])
    print(f"  {row['Sortie']} (Alphanumeric: '{source_alpha}')")

# Check if our test cases were successfully matched
print(f"\n🧪 TEST CASE VALIDATION:")
test_cases = [
    ("104W_035C", "104W_C_0035"),
    ("31_4788", "US31_4788"), 
    ("7_156A", "US7_0156_A")
]

for source, expected_target in test_cases:
    match_row = merged_df[merged_df['Sortie'] == source]
    if len(match_row) > 0:
        actual_target = match_row.iloc[0]['Sortie_BilderbestellungenFluginfos']
        similarity = match_row.iloc[0]['Similarity_Percentage']
        status = "✅" if pd.notna(actual_target) else "❌"
        print(f"  {status} {source} -> {actual_target} ({similarity:.1f}%)")
    else:
        print(f"  ❌ {source} not found in merged data")

# Save the enhanced DataFrame with all results
output_enhanced_path = os.path.join(directory_path, 'Merged_Enhanced.xlsx')
merged_df.to_excel(output_enhanced_path, index=False)
print(f"\n💾 Enhanced DataFrame saved to: {output_enhanced_path}")

# Generate detailed analysis report
analysis_report_path = os.path.join(directory_path, 'Matching_Analysis_Report.xlsx')
with pd.ExcelWriter(analysis_report_path) as writer:
    # Main data
    merged_df.to_excel(writer, sheet_name='All_Data', index=False)
    
    # Summary statistics
    summary_data = {
        'Metric': ['Total Rows', 'Rows with Matches', 'Rows without Matches', 
                  'Average Similarity', 'Very High Confidence', 'High Confidence',
                  'Medium Confidence', 'Low Confidence', 'Very Low Confidence'],
        'Value': [total_rows, total_matches, no_matches, avg_similarity,
                 confidence_stats['Very High'], confidence_stats['High'],
                 confidence_stats['Medium'], confidence_stats['Low'], 
                 confidence_stats['Very Low']]
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name='Summary', index=False)
    
    # High confidence matches
    high_confidence = merged_df[merged_df['Match_Confidence'].isin(['Very High', 'High'])]
    high_confidence.to_excel(writer, sheet_name='High_Confidence_Matches', index=False)
    
    # Non-matches for review
    non_matches_df = merged_df[~merged_df['Match_Found']]
    non_matches_df.to_excel(writer, sheet_name='Non_Matches', index=False)

print(f"💾 Detailed analysis report saved to: {analysis_report_path}")
print("="*90)
print("✅ MATCHING ANALYSIS COMPLETED SUCCESSFULLY!")


📈 COMPREHENSIVE MATCHING RESULTS ANALYSIS
📊 OVERALL STATISTICS:
Total rows processed: 16,991
Rows with matches: 16,990 (100.0%)
Rows without matches: 1 (0.0%)
Average similarity for matches: 89.5%

🎯 MATCH CONFIDENCE DISTRIBUTION:
--------------------------------------------------
  Very High : 7,840 matches ( 46.1% of matches)
  High      : 6,390 matches ( 37.6% of matches)
  Medium    : 2,585 matches ( 15.2% of matches)
  Low       :   121 matches (  0.7% of matches)
  Very Low  :    54 matches (  0.3% of matches)
  No Match  :     1 rows

📈 SIMILARITY SCORE DISTRIBUTION:
  <50%    :     0 rows
  50-60%  :    55 rows
  60-70%  :   120 rows
  70-80%  : 2,734 rows
  80-90%  : 6,260 rows
  90-100% : 7,821 rows

⭐ TOP 10 HIGHEST SIMILARITY MATCHES:
   1. 106G-1657 -> 106G_1657
      Similarity: 100.0%, Confidence: Very High
      Alphanumeric: '106G1657' vs '106G1657'
   2. 106G-1657 -> 106G_1657
      Similarity: 100.0%, Confidence: Very High
      Alphanumeric: '106G1657' vs '106G1657